# Geodaten der Stadt Zürich via WFS beziehen

Dieses Notebook zeigt, wie Geodaten aus dem [Geoportal der Stadt Zürich](https://www.stadt-zuerich.ch/geodaten/)
über einen **WFS** (Web Feature Service) bezogen und weiterverarbeitet werden können — als Tabelle,
als Karte, mit Pagination und mit serverseitigen Filtern.

Ein WFS-Dienst liefert **Vektor-Geodaten**. Ein WFS enthält üblicherweise mehrere *Layer*
(Feature-Typen) mit unterschiedlichen Inhalten. Die URL zum WFS-Dienst findet man im
[OGD-Katalog](https://data.stadt-zuerich.ch/dataset?res_format=WFS) bei der jeweiligen Ressource
oder direkt auf dem Geoportal.

Als Beispiel dient der Datensatz
[**Statistische Quartiere**](https://data.stadt-zuerich.ch/dataset/geo_statistische_quartiere)
(34 statistische Quartiere, Lizenz CC0).

Verwendet wird durchgehend **WFS 1.1.0** — so, wie die Dienste im OGD-Katalog publiziert sind.

> Der Server beherrscht auch noch das ältere WFS 1.0.0. Die Datenantworten sind in beiden
> Versionen identisch, das Capabilities-Dokument ist aber anders aufgebaut. Wer altem Code
> begegnet oder beide Versionen bedienen muss, findet die Details im Notebook
> [Geoportal-Beispiele-WFS110-vs-WFS100.ipynb](geoportal-beispiele-wfs110-vs-wfs100.ipynb).

Mit Renku oder Colab kann das Notebook interaktiv im Browser gestartet werden:

[![Python - renku](https://renkulab.io/renku-badge.svg)](https://renkulab.io/p/opendatazurich/github-jupyter/sessions/01M3PPQJDDQZ4YD7CD7DZ5S5ER/start?GITHUB_FILEPATH=opendatazurich/opendatazurich.github.io/blob/master/geoportal/Geoportal-Beispiele.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opendatazurich/opendatazurich.github.io/blob/master/geoportal/Geoportal-Beispiele.ipynb)

In [1]:
%pip install --quiet geopandas requests folium

Note: you may need to restart the kernel to use updated packages.


In [2]:
import xml.etree.ElementTree as ET

import folium
import geopandas
import requests

print("geopandas", geopandas.__version__, "| folium", folium.__version__)

geopandas 1.1.1 | folium 0.20.0


In [3]:
wfs_url = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere"

# Die WFS-Version wird an einer Stelle definiert und überall mitgegeben.
WFS_VERSION = "1.1.0"

# Namespaces des Capabilities-Dokuments
namespaces = {
    'wfs': 'http://www.opengis.net/wfs',
    'ows': 'http://www.opengis.net/ows',
}

## GetCapabilities

Welche Layer, Formate und Operationen ein WFS-Server anbietet, verrät der `GetCapabilities`-Aufruf.

In [4]:
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetCapabilities',
})
r.raise_for_status()

root = ET.fromstring(r.content)
print("Version:", root.get('version'))
print()
print(r.text[:700])

Version: 1.1.0

<WFS_Capabilities xsi:schemaLocation="http://www.opengis.net/wfs http://schemas.opengis.net/wfs/1.1.0/wfs.xsd" xmlns="http://www.opengis.net/wfs" xmlns:gml="http://www.opengis.net/gml" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:ogc="http://www.opengis.net/ogc" version="1.1.0" xmlns:ows="http://www.opengis.net/ows" updateSequence="0" xmlns:xlink="http://www.w3.org/1999/xlink">
 <ows:ServiceIdentification>
  <ows:Title>Untitled</ows:Title>
  <ows:ServiceType>WFS</ows:ServiceType>
  <ows:ServiceTypeVersion>1.1.0</ows:ServiceTypeVersion>
  <ows:Fees>None</ows:Fees>
  <ows:AccessConstraints>None</ows:AccessConstraints>
 </ows:ServiceIdentification>
 <ows:ServiceProvider/>
 <ows:O


### Die Layer auslesen

Jeder Layer steht als `FeatureType` in der `FeatureTypeList`. Interessant sind `Name` und die
Koordinatensysteme (`DefaultSRS`, `OtherSRS`).

In [5]:
layers = {}
for feature_type in root.findall('wfs:FeatureTypeList/wfs:FeatureType', namespaces):
    name = feature_type.findtext('wfs:Name', namespaces=namespaces)
    layers[name] = {
        'default_srs': feature_type.findtext('wfs:DefaultSRS', namespaces=namespaces) or None,
        'other_srs': [e.text for e in feature_type.findall('wfs:OtherSRS', namespaces)],
    }

layers

{'adm_statistische_quartiere_b_p': {'default_srs': None,
  'other_srs': ['EPSG:4326', 'EPSG:3857']},
 'adm_statistische_quartiere_map': {'default_srs': None,
  'other_srs': ['EPSG:4326', 'EPSG:3857']},
 'adm_statistische_quartiere_v': {'default_srs': None,
  'other_srs': ['EPSG:4326', 'EPSG:3857']}}

> **Beachten:** `default_srs` ist leer — dieser Server füllt `<DefaultSRS>` nicht aus.
> Das Koordinatensystem lässt sich also *nicht* aus den Capabilities ablesen, man legt es beim
> Abruf selbst fest (siehe Abschnitt *Koordinatensystem* weiter unten).

Der Layer **`adm_statistische_quartiere_map`** ist der kartografisch aufbereitete Layer und damit
für Visualisierungen der passende:

> Dieser Layer kann für Datenvisualisierungen und kartographische Darstellungen verwendet werden.
> Er enthält die vollständige Attributtabelle und ist generalisiert, sprich: die Geometrie wurde
> vereinfacht und entspricht nicht der – aus Sicht der amtlichen Vermessung – korrekten Lage.

Für lagegenaue Auswertungen ist er deshalb nicht geeignet — dafür gibt es
`adm_statistische_quartiere_v`.

Die Layer sind im [OGD-Katalog](https://data.stadt-zuerich.ch/dataset/geo_statistische_quartiere)
beschrieben, die kompletten Metadaten inklusive aller Attribute liegen auf
[geocat.ch](https://www.geocat.ch/geonetwork/srv/ger/catalog.search#/metadata/fd1a94fe-4bd4-4a40-99af-8b859dfe82a7).

Zum Verknüpfen mit anderen Datensätzen eignen sich `qnr` (Statistisches Quartier) bzw. `knr`
(Stadtkreis). Die `objectid` ist laut Metadaten ein interner Identifikator und **nicht stabil
über Zeit** — sie taugt nicht als Schlüssel.

In [6]:
layer = 'adm_statistische_quartiere_map'

### Welche Ausgabeformate bietet GetFeature?

Die unterstützten Formate stehen in den `OperationsMetadata`.

In [7]:
output_formats = []
for operation in root.findall('ows:OperationsMetadata/ows:Operation', namespaces):
    if operation.get('name') != 'GetFeature':
        continue
    for parameter in operation.findall('ows:Parameter', namespaces):
        if parameter.get('name') == 'outputFormat':
            output_formats = [v.text for v in parameter.findall('ows:Value', namespaces)]

output_formats

['text/xml; subtype=gml/2.1.2',
 'text/xml; subtype=gml/3.1.1',
 'application/vnd.geo+json']

`application/vnd.geo+json` ist das GeoJSON-Format. Der Kurzname **`GeoJSON`** wird vom Server
ebenfalls akzeptiert und ist bequemer zu schreiben — beide liefern dasselbe Resultat.

## Daten via GetFeature als GeoJSON laden

In [8]:
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
})
r.raise_for_status()
stat_quarter_geo = r.json()

print("Content-Type:   ", r.headers['content-type'])
print("Anzahl Features:", len(stat_quarter_geo['features']))
print("Attribute:      ", list(stat_quarter_geo['features'][0]['properties']))
print("Erstes Feature: ", stat_quarter_geo['features'][0]['properties'])

Content-Type:    application/vnd.geo+json; charset=utf-8
Anzahl Features: 34
Attribute:       ['geometrie_gdo', 'kname', 'knr', 'objectid', 'qname', 'qnr']
Erstes Feature:  {'geometrie_gdo': None, 'kname': 'Kreis 7', 'knr': 7, 'objectid': 1, 'qname': 'Hirslanden', 'qnr': 73}


Derselbe Aufruf als Link, zum Ausprobieren im Browser:
[GetFeature als GeoJSON](https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere?service=WFS&version=1.1.0&request=GetFeature&typename=adm_statistische_quartiere_map&outputFormat=GeoJSON)

### Koordinatensystem: WGS 84 oder LV95?

* **Ohne** `srsName` kommt das GeoJSON in **EPSG:4326** (WGS 84, Grad) — so verlangt es die
  GeoJSON-Spezifikation. Das ist das Format, das Leaflet und folium direkt verstehen.
* **Mit** `srsName=EPSG:2056` kommt LV95 (Meter) — passend zu den übrigen städtischen Geodaten
  und nötig, wenn man in Metern rechnen will (Flächen, Distanzen, Puffer).

### Daten in GeoPandas als Tabelle

In [9]:
data = geopandas.GeoDataFrame.from_features(stat_quarter_geo, crs="EPSG:4326")
data.head(3)

,geometry,geometrie_gdo,kname,knr,objectid,qname,qnr
0,"POLYGON ((8.5567 47.36406, 8.55682 47.3641, 8....",None,Kreis 7,7,1,Hirslanden,73
1,"POLYGON ((8.5567 47.36405, 8.55672 47.36403, 8...",None,Kreis 8,8,2,Weinegg,83
2,"POLYGON ((8.55423 47.36456, 8.55426 47.36452, ...",None,Kreis 8,8,3,Mühlebach,82


> **Nicht mehr verwenden:** `crs={'init': 'EPSG:4326'}`. Diese Dict-Schreibweise ist seit
> pyproj 2 veraltet und führt in aktuellen Versionen zu einem Fehler. Korrekt ist der
> String `crs="EPSG:4326"`.

Dieselben Daten in LV95, um zum Beispiel Flächen in Hektaren zu berechnen:

In [10]:
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'srsName': 'EPSG:2056',
})
r.raise_for_status()

data_lv95 = geopandas.GeoDataFrame.from_features(r.json(), crs="EPSG:2056")
data_lv95['flaeche_ha'] = (data_lv95.area / 10_000).round(1)
data_lv95[['qname', 'kname', 'knr', 'qnr', 'flaeche_ha']] \
    .sort_values('flaeche_ha', ascending=False) \
    .head(5)

,qname,kname,knr,qnr,flaeche_ha
29,Altstetten,Kreis 9,9,92,745.5
18,Höngg,Kreis 10,10,101,698.4
20,Affoltern,Kreis 11,11,111,604.4
4,Wollishofen,Kreis 2,2,21,578.1
27,Friesenberg,Kreis 3,3,33,529.8


### Pagination

Wenn ein WFS sehr viele Features zurückliefert, kann man die Daten portionenweise beziehen.
Dafür dienen `startIndex` und `maxFeatures`.

> Der Parameter heisst `maxFeatures`. Das aus WFS 2.0.0 bekannte `count` ignoriert dieser
> Server — er würde stillschweigend *alle* Features liefern.

Wie viele Features es insgesamt gibt, verrät ein `GetFeature`-Request mit `resultType=hits`:
er liefert eine leere FeatureCollection, dafür mit dem Attribut `numberOfFeatures`.

In [11]:
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'resultType': 'hits',
})
r.raise_for_status()

hits = ET.fromstring(r.content)
print("Total Features:", hits.get('numberOfFeatures'))

Total Features: 34


In [12]:
# Ab Index 5 (die ersten fünf werden übersprungen) genau ein Feature holen
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'startIndex': 5,
    'maxFeatures': 1,
})
r.raise_for_status()

one_page = geopandas.GeoDataFrame.from_features(r.json(), crs="EPSG:4326")
one_page[['qname', 'kname', 'knr', 'qnr']]

,qname,kname,knr,qnr
0,Leimbach,Kreis 2,2,23


### Daten in einer Karte darstellen

Als Hintergrund dient der WMS
[Basiskarte Zürich Raster Grau](https://data.stadt-zuerich.ch/dataset/geo_basiskarte_zuerich_raster_grau).

> **Wichtig:** Der Layer-Name im WMS ist **`wmslayers`** (so steht er in den WMS-Capabilities) —
> *nicht* der Dienstname `Basiskarte_Zuerich_Raster_Grau`. Mit einem falschen Layer-Namen
> antwortet der Server mit HTTP 500 und die Karte bleibt leer.

In [13]:
def basiskarte(location=(47.38, 8.53), zoom_start=12):
    """Folium-Karte mit der städtischen Basiskarte als Hintergrund."""
    m = folium.Map(location=location, zoom_start=zoom_start, tiles=None)
    folium.raster_layers.WmsTileLayer(
        url='https://www.ogd.stadt-zuerich.ch/wms/geoportal/Basiskarte_Zuerich_Raster_Grau',
        layers='wmslayers',          # <-- korrekter Layer-Name
        name='Zürich - Basiskarte',
        fmt='image/png',
        overlay=False,
        control=False,
        autoZindex=False,
        attr='&copy; Stadt Zürich, Open Government Data',
    ).add_to(m)
    return m

In [14]:
m = basiskarte()
folium.GeoJson(
    stat_quarter_geo,
    name='Statistische Quartiere',
    style_function=lambda feature: {'color': '#0f05a0', 'weight': 1.5, 'fillOpacity': 0.07},
    tooltip=folium.GeoJsonTooltip(fields=['qname', 'kname'], aliases=['Quartier', 'Kreis']),
).add_to(m)
m

## Daten filtern

Filtern lohnt sich: der Server schickt dann nur die benötigten Features, statt dass man alles
herunterlädt und lokal wieder wegwirft.

### Mit Attributen filtern: OGC-Filter

Attributfilter werden als XML formuliert und über den Parameter `Filter` an `GetFeature` übergeben.

> ⚠️ **Fallstrick beim GET-Request:** Enthält der Filter in der URL eine **`xmlns`-Deklaration**,
> wird die Anfrage von der Web Application Firewall mit **HTTP 403** abgewiesen. In Python
> scheitert dann `r.json()` mit
> `JSONDecodeError: Expecting value: line 1 column 1 (char 0)`, weil statt GeoJSON eine
> HTML-Fehlerseite zurückkommt.
>
> Zwei Lösungen:
>
> 1. Beim GET-Request die `xmlns`-Deklarationen **weglassen** — der QGIS-Server versteht den
>    Filter auch ohne (nächste Zelle).
> 2. Den Request als **POST** senden; dort sind vollständige Namespaces problemlos
>    (siehe weiter unten).
>
> Eine XML-Deklaration (`<?xml version="1.0"?>`) ist unkritisch — nur `xmlns` ist das Problem.

In [15]:
# Filter für Kreis 7 -- bewusst OHNE xmlns-Deklaration
filter_xml = """
<Filter>
    <PropertyIsEqualTo>
        <PropertyName>knr</PropertyName>
        <Literal>7</Literal>
    </PropertyIsEqualTo>
</Filter>
"""

r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'Filter': filter_xml,
})
r.raise_for_status()

filtered_data = geopandas.GeoDataFrame.from_features(r.json(), crs="EPSG:4326")
filtered_data[['qname', 'kname', 'knr', 'qnr']]

,qname,kname,knr,qnr
0,Hirslanden,Kreis 7,7,73
1,Hottingen,Kreis 7,7,72
2,Witikon,Kreis 7,7,74
3,Fluntern,Kreis 7,7,71


#### Dasselbe als POST-Request

Beim POST wird der komplette `GetFeature`-Request als XML im Body übergeben. Hier sind Namespaces
ausdrücklich erlaubt, und man ist nicht an die maximale URL-Länge gebunden — für grosse oder
verschachtelte Filter die robustere Variante.

In [16]:
post_body = """<?xml version="1.0" encoding="UTF-8"?>
<wfs:GetFeature service="WFS" version="1.1.0" outputFormat="GeoJSON"
                xmlns:wfs="http://www.opengis.net/wfs"
                xmlns:ogc="http://www.opengis.net/ogc">
  <wfs:Query typeName="adm_statistische_quartiere_map">
    <ogc:Filter>
      <ogc:PropertyIsEqualTo>
        <ogc:PropertyName>knr</ogc:PropertyName>
        <ogc:Literal>7</ogc:Literal>
      </ogc:PropertyIsEqualTo>
    </ogc:Filter>
  </wfs:Query>
</wfs:GetFeature>
"""

r = requests.post(wfs_url,
                  data=post_body.encode('utf-8'),
                  headers={'Content-Type': 'text/xml'})
r.raise_for_status()

post_data = geopandas.GeoDataFrame.from_features(r.json(), crs="EPSG:4326")
post_data[['qname', 'kname', 'knr', 'qnr']]

,qname,kname,knr,qnr
0,Hirslanden,Kreis 7,7,73
1,Hottingen,Kreis 7,7,72
2,Witikon,Kreis 7,7,74
3,Fluntern,Kreis 7,7,71


#### EXP_FILTER (QGIS-WFS-Server)

Zusätzlich unterstützt der verwendete QGIS-WFS-Server den herstellerspezifischen Parameter
`EXP_FILTER` mit einer SQL-ähnlichen Syntax — deutlich kompakter als OGC-Filter-XML.

Mögliche Ausdrücke:

* `field = 10`
* `field_a > 10`
* `field_b > 10 AND field_c <= 7`
* `field_d LIKE 'H%'`

In [17]:
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'EXP_FILTER': "qname LIKE 'H%'",
})
r.raise_for_status()

like_data = geopandas.GeoDataFrame.from_features(r.json(), crs="EPSG:4326")
like_data[['qname', 'kname', 'knr', 'qnr']]

,qname,kname,knr,qnr
0,Hirslanden,Kreis 7,7,73
1,Hottingen,Kreis 7,7,72
2,Hard,Kreis 4,4,44
3,Höngg,Kreis 10,10,101
4,Hochschulen,Kreis 1,1,12
5,Hirzenbach,Kreis 12,12,123


### Geografisch filtern

Die Daten lassen sich auch räumlich einschränken. Am einfachsten geht das mit dem
Standard-Parameter **`bbox`** (Bounding Box) in der Form `minx,miny,maxx,maxy[,CRS]`.

Im folgenden Beispiel holen wir aus dem
[Baumkataster](https://data.stadt-zuerich.ch/dataset/geo_baumkataster) alle Bäume der Gattung
*Prunus* (Kirschen, Pflaumen, Zwetschgen) im Gebiet rund um den Kreis 7 — also eine Kombination
aus geografischem und Attributfilter.

In [18]:
baumkataster_wfs = 'https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Baumkataster'
baum_layer = 'baumkataster_baumstandorte'

# Bounding Box in LV95 (EPSG:2056): Gebiet Kreis 7
bbox_kreis7 = '2684454.0423,1245344.4639,2687039.4349,1247714.1562,EPSG:2056'

r = requests.get(baumkataster_wfs, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': baum_layer,
    'outputFormat': 'GeoJSON',
    'bbox': bbox_kreis7,
    'EXP_FILTER': "baumgattunglat = 'Prunus'",
})
r.raise_for_status()
tree_geo = r.json()

tree_data = geopandas.GeoDataFrame.from_features(tree_geo, crs="EPSG:4326")
print("Anzahl Bäume:", len(tree_data))
tree_data[['baumnamedeu', 'baumartlat', 'quartier', 'pflanzjahr', 'kronendurchmesser']].head(5)

Anzahl Bäume: 580


,baumnamedeu,baumartlat,quartier,pflanzjahr,kronendurchmesser
0,"Sammelbegriff für Zwetschgen, Pflaume, Reneklo...",domestica,Weinegg,2000.0,8
1,"Sammelbegriff für Zwetschgen, Pflaume, Reneklo...",domestica,Hirslanden,1950.0,8
2,"Sammelbegriff für Zwetschgen, Pflaume, Reneklo...",domestica,Hirslanden,1950.0,8
3,"Vogel-Kirsche, Süss-Kirsche",avium,Hirslanden,2000.0,8
4,"Sammelbegriff für Zwetschgen, Pflaume, Reneklo...",domestica,Hirslanden,2000.0,8


Alternativ als OGC-Filter mit `BBOX` und `And` — wiederum ohne `xmlns`, damit der GET-Request
nicht blockiert wird:

In [19]:
geo_filter = """
<Filter>
  <And>
    <BBOX>
      <PropertyName>geometry</PropertyName>
      <gml:Box srsName="EPSG:2056">
        <gml:coordinates cs="," ts=" ">2684454.0423,1245344.4639 2687039.4349,1247714.1562</gml:coordinates>
      </gml:Box>
    </BBOX>
    <PropertyIsEqualTo>
      <PropertyName>baumgattunglat</PropertyName>
      <Literal>Prunus</Literal>
    </PropertyIsEqualTo>
  </And>
</Filter>
"""

r = requests.get(baumkataster_wfs, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': baum_layer,
    'outputFormat': 'GeoJSON',
    'Filter': geo_filter,
})
r.raise_for_status()
print("Anzahl Bäume:", len(r.json()['features']))

Anzahl Bäume: 580


In [20]:
tree_map = basiskarte(location=(47.365, 8.555), zoom_start=14)
folium.GeoJson(
    tree_geo,
    name='Prunus-Bäume',
    marker=folium.CircleMarker(radius=4, color='#8b1a4a', fill=True,
                               fill_opacity=0.8, weight=1),
    tooltip=folium.GeoJsonTooltip(
        fields=['baumnamedeu', 'quartier', 'pflanzjahr'],
        aliases=['Baum', 'Quartier', 'Pflanzjahr'],
    ),
).add_to(tree_map)
tree_map

## Zusammenfassung der Stolpersteine

| Problem | Ursache | Lösung |
|---|---|---|
| Koordinatensystem aus den Capabilities ist leer | Server füllt `DefaultSRS` nicht aus | CRS selbst setzen: GeoJSON ist EPSG:4326, LV95 via `srsName=EPSG:2056` |
| `JSONDecodeError` beim Filtern | `xmlns` im GET-Parameter → HTTP 403 der Firewall | `xmlns` weglassen oder POST verwenden |
| Fehler bei `crs={'init': ...}` | seit pyproj 2 veraltet | `crs="EPSG:4326"` |
| Karte bleibt leer | falscher WMS-Layer-Name → HTTP 500 | `layers='wmslayers'` |
| `count` wirkt nicht | `count` ist WFS 2.0.0 | `maxFeatures` verwenden |

## Weiterführende Links

* [Geoportal der Stadt Zürich](https://www.stadt-zuerich.ch/geodaten/)
* [OGD-Katalog: alle Datensätze mit WFS](https://data.stadt-zuerich.ch/dataset?res_format=WFS)
* [README dieses Verzeichnisses](README.md) — WMS/WMTS in Leaflet einbinden
* [Geoportal-Beispiele-WFS110-vs-WFS100.ipynb](geoportal-beispiele-wfs110-vs-wfs100.ipynb)
  — Unterschiede zwischen WFS 1.1.0 und 1.0.0
* [OGD-Werkstatt der Stadt Zürich](https://www.stadt-zuerich.ch/de/politik-und-verwaltung/statistik-und-daten/open-government-data/werkstatt.html)
  — Geodaten mit GIS-Software
* [Esri: Communicating with a WFS service in a web browser](https://enterprise.arcgis.com/de/server/10.3/publish-services/linux/communicating-with-a-wfs-service-in-a-web-browser.htm)